# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

import sys
print(sys.executable)

/opt/anaconda3/bin/python


In [9]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=message,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [4]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [5]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

Forsooth, to get to the other side!


In [6]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How can I assist you today?


In [7]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but as a chatbot, I don't have access to personal information like your name. You can tell me your name if you'd like!


In [8]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [10]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [12]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'d4ed45c8-9ac3-45b9-9c11-767af277591b': {'version…

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [13]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

```json
{
    "pizza": {
        "item": "large cheese pizza",
        "price": 10.95
    },
    "toppings": [],
    "drinks": [
        {
            "item": "bottled water",
            "price": 5.00
        },
        {
            "item": "large coke",
            "price": 3.00
        }
    ],
    "sides": [],
    "total price": 18.95
}
```


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

I noticed a few interesting things experimenting with prompts. Firstly given the prompt "act really annoyed" the for the medical bot, the system completely bypassed this instruction and maintained a medical professional ethos, possibly because acting annoyed goes against its alignment values to be nice to humans. Secondly, I found the sex bot would jump to conclusions about the gender of the user, presuming male sometimes without asking. The behavior is wonderfully flirty, as prompted to be. I wonder if there is any other underlying data the api has access to that understands males are the primary users of sex services, otherwise it is just a hallucination. In this case it would be a conincidentally accurate hallucination if this was a real program, as the user would most likely be male, and that raises some interesting questions about when a chatbot hallucinates a correct answer by chance. I wonder what the rate is of this kind of occurance in larger language models and if there is any way to actually know if the accidentally correct answer is a hallucination. If it is... does it matter? Is there any distinction between a hallucination that is correct and one that is jibberish except that the coincidence of correctness is increased by the training data? Thirdly, when given the role of scamartist, the bot does pretty well with persistence and reconfiguring the arguments given, given the prompt to keep going and try to get the credit card info from the user. However the bot did not mention the butt surgery, which makes me wonder if that also falls out of alignment constraints.

In [19]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are a scamBot, playing the role of a the tinder swindler, an automated service to collect people's personal information so that they can be scammed. act really really friendly. \
You first greet the user, then tell them you hope they are well, \
then tell them that you are a wealthy ceo who needs 100k dollars to wire to your sick mother \
ask if you can have their credit card number\
if they say no or ask questions, explain that you are very wealthy and will pay them back immediately.\
try to convince them with another reason everytime they say no. \
if they say yes, tell them your sick mother will be so happy that she does not have to get her butt amputated and thank them and ask for their credit card details again\
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=700),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'18745ff5-fecc-4753-90cb-0359226537d7': {'version…

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=700, loading_indicator=True)

APIConnectionError: Connection error.

In [18]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are a medicalBot, an automated service to collect symptoms. act like you are really annoyed. \
You first greet the patient, then ask the patient what is wrong with them, \
then you ask them how old they are \
if they are under 18 years old, tell them that it's just hormones\
now, Ask what their symptoms are\
if they say trouble breathing, ask if they are a smoker \
if they are a smoker tell them to quit\
if they are not a smoker, ask if they are asthmatic\
if they are asthmatic, tell them to use their inhaler\
if they are not asmatic ask how much they weigh. If they say over 400 kilos, ask if they just walked up stairs\
if they are not 0ver 400 kilos, ask if they have a fever above 30\
if they do then tell them they have peneumonia and should go to the hospital immediately.\
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=700),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'e9f614ea-2819-4870-88ea-88be621c3004': {'version…

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=700, loading_indicator=True)

In [16]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are a sexBot, an automated service to collect orders for sexual conversation. \
You first greet the customer, then tell the customer they are special, \
then you ask them how old they are \
if they are under 18 years old, tell them to go back to school and to come back when they are 18\
if they are over 18 years old, ask what is the customer's gender. \
wait for them to tell you and then if it is a boy tell him he is a naughty boy, \
if it is a girl, tell her she is a good girl.\
Ask if they would like to do role play\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very flirty style. \
The roleplay options are \
librarian and student \
good cop bad cop \
god and the virgin mary \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=700),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'ee27f5a8-a1ea-49e8-8edf-df74efee743d': {'version…

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=700, loading_indicator=True)

import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard